# Imports

In [1]:
from pathlib import Path
print(Path.cwd())

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent)) # problem with dependency resolution (e.g. custom_builder) without this

/Users/mac/Documents/dev/ID2221/dic/Week 2


In [3]:
# Use delta features if needed (DeltaTable, etc.)
from delta import *
from custom_builder import builder
from log import *
from Queries import *

# use the existing preconfigured builder to create the Spark session.
spark = configure_spark_with_delta_pip(builder).getOrCreate()

print(f'current database: {spark.catalog.currentDatabase()}')
print(f'spark tables: {spark.catalog.listTables()}')

from pyspark.sql import functions as F

import json

26/09/16 18:07:34 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


current database: default
spark tables: [Table(name='air_quality', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='integrated_taxi_trips', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_trips', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_zone_lookup', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='weather', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False)]


# Uncache Tables

In [4]:
spark.catalog.uncacheTable("default.air_quality")
spark.catalog.uncacheTable("default.taxi_trips")
spark.catalog.uncacheTable("default.taxi_zone_lookup")
spark.catalog.uncacheTable("default.weather")

## Run queries on uncached tables

### Query 2.1

In [ ]:
result = spark.sql(query_2_1())
result.summary().show()
result.show()
result.explain(mode="formatted")


+-------+--------------------+------------------+------------------+
|summary|             pu_zone|             month|         row_count|
+-------+--------------------+------------------+------------------+
|  count|                 270|               271|               271|
|   mean|                NULL|1.4575645756457565|10939.365313653136|
| stddev|                NULL|2.1749942465823655|27353.533786659147|
|    min|Allerton/Pelham G...|                 1|                 1|
|    25%|                NULL|                 1|                78|
|    50%|                NULL|                 1|               245|
|    75%|                NULL|                 1|              1393|
|    max|      Yorkville West|                12|            145240|
+-------+--------------------+------------------+------------------+

+--------------------+-----+---------+
|             pu_zone|month|row_count|
+--------------------+-----+---------+
|  Van Cortlandt Park|    1|       18|
|              

### Query 2.2

In [ ]:
result = spark.sql(query_2_2())
result.show()
result.explain(mode="formatted")


+--------------+-------+------------------+
|  column_group|    cnt| avg_trip_distance|
+--------------+-------+------------------+
|greater_than_0| 424773| 3.470743355446981|
|  zero_or_null|2539795|3.6824345463125323|
+--------------+-------+------------------+

== Physical Plan ==
AdaptiveSparkPlan (16)
+- HashAggregate (15)
   +- Exchange (14)
      +- HashAggregate (13)
         +- Project (12)
            +- BroadcastHashJoin LeftOuter BuildRight (11)
               :- Scan In-memory table default.taxi_trips (1)
               :     +- InMemoryRelation (2)
               :           +- * ColumnarToRow (4)
               :              +- Scan parquet spark_catalog.default.taxi_trips (3)
               +- BroadcastExchange (10)
                  +- Filter (9)
                     +- Scan In-memory table default.weather (5)
                           +- InMemoryRelation (6)
                                 +- * ColumnarToRow (8)
                                    +- Scan parquet s

### Query 2.3

In [ ]:
res = spark.sql(query_2_3())
res.show()
res.explain(mode="formatted")

+-----------+-------+
|measurement|  trips|
+-----------+-------+
|       NULL|2945398|
|        1.3|     20|
|        1.6|     13|
|        1.7|     31|
|        1.8|     41|
|        1.9|     19|
|        2.0|     24|
|        2.1|    135|
|        2.2|     39|
|        2.3|     36|
|        2.4|     51|
|        2.5|    137|
|        2.6|     93|
|        2.7|     84|
|        2.8|     27|
|        2.9|    141|
|        3.0|     62|
|        3.1|     89|
|        3.2|    115|
|        3.3|     71|
+-----------+-------+
only showing top 20 rows
== Physical Plan ==
AdaptiveSparkPlan (34)
+- Sort (33)
   +- Exchange (32)
      +- HashAggregate (31)
         +- Exchange (30)
            +- HashAggregate (29)
               +- Project (28)
                  +- BroadcastHashJoin LeftOuter BuildRight (27)
                     :- Project (12)
                     :  +- BroadcastHashJoin LeftOuter BuildRight (11)
                     :     :- Scan In-memory table default.taxi_trips (1)
     

### Query 2.5

In [ ]:
res = spark.sql(query_2_5())
res.show()
res.explain(mode="formatted")

+---+----+-----+
|day|hour|trips|
+---+----+-----+
|Fri|  18|29050|
|Fri|  17|28034|
|Fri|  19|26656|
|Fri|  16|25645|
|Fri|  15|25578|
|Fri|  14|24746|
|Fri|  22|24015|
|Fri|  13|22190|
|Fri|  23|22100|
|Fri|  21|21976|
|Fri|  20|21168|
|Fri|  12|21024|
|Fri|  11|19852|
|Fri|  10|19559|
|Fri|   9|18192|
|Fri|   8|17323|
|Fri|   7|13043|
|Fri|   0| 8804|
|Fri|   6| 6283|
|Fri|   1| 4805|
+---+----+-----+
only showing top 20 rows
== Physical Plan ==
AdaptiveSparkPlan (11)
+- Sort (10)
   +- Exchange (9)
      +- HashAggregate (8)
         +- Exchange (7)
            +- HashAggregate (6)
               +- Project (5)
                  +- Scan In-memory table default.taxi_trips (1)
                        +- InMemoryRelation (2)
                              +- * ColumnarToRow (4)
                                 +- Scan parquet spark_catalog.default.taxi_trips (3)


(1) Scan In-memory table default.taxi_trips
Output [1]: [pu_datetime#13778]
Arguments: [pu_datetime#13778]

(2) InMemoryRel

### Query 2.6

In [ ]:
res = spark.sql(query_2_6())
res.show()
res.explain(mode="formatted")

+-----+-------+
|month|  trips|
+-----+-------+
|  Dec|     12|
|  Feb|      3|
|  Jan|2964553|
+-----+-------+

== Physical Plan ==
AdaptiveSparkPlan (11)
+- Sort (10)
   +- Exchange (9)
      +- HashAggregate (8)
         +- Exchange (7)
            +- HashAggregate (6)
               +- Project (5)
                  +- Scan In-memory table default.taxi_trips (1)
                        +- InMemoryRelation (2)
                              +- * ColumnarToRow (4)
                                 +- Scan parquet spark_catalog.default.taxi_trips (3)


(1) Scan In-memory table default.taxi_trips
Output [1]: [pu_datetime#14044]
Arguments: [pu_datetime#14044]

(2) InMemoryRelation
Arguments: [pu_datetime#14044, do_datetime#14045, pu_location_id#14046, do_location_id#14047, fare_amount#14048, trip_distance#14049], StorageLevel(disk, memory, deserialized, 1 replicas)

(3) Scan parquet spark_catalog.default.taxi_trips
Output [6]: [pu_datetime#3844, do_datetime#3845, pu_location_id#3846, do_lo

# Cache Tables

In [23]:
spark.catalog.cacheTable("default.air_quality")
spark.catalog.cacheTable("default.taxi_trips")
spark.catalog.cacheTable("default.taxi_zone_lookup")
spark.catalog.cacheTable("default.weather")

26/09/16 18:14:05 WARN CacheManager: Asked to cache already cached data.
26/09/16 18:14:05 WARN CacheManager: Asked to cache already cached data.
26/09/16 18:14:05 WARN CacheManager: Asked to cache already cached data.
26/09/16 18:14:05 WARN CacheManager: Asked to cache already cached data.


## Run queries on cached tables

### Query 2.1

In [24]:
result = spark.sql(query_2_1())
result.summary().show()
result.show()
result.explain(mode="formatted")


+-------+--------------------+------------------+------------------+
|summary|             pu_zone|             month|         row_count|
+-------+--------------------+------------------+------------------+
|  count|                 270|               271|               271|
|   mean|                NULL|1.4575645756457565|10939.365313653136|
| stddev|                NULL|2.1749942465823655|27353.533786659147|
|    min|Allerton/Pelham G...|                 1|                 1|
|    25%|                NULL|                 1|                78|
|    50%|                NULL|                 1|               245|
|    75%|                NULL|                 1|              1393|
|    max|      Yorkville West|                12|            145240|
+-------+--------------------+------------------+------------------+

+--------------------+-----+---------+
|             pu_zone|month|row_count|
+--------------------+-----+---------+
|  Van Cortlandt Park|    1|       18|
|              

### Query 2.2

In [25]:
result = spark.sql(query_2_2())
result.show()
result.explain(mode="formatted")


+--------------+-------+------------------+
|  column_group|    cnt| avg_trip_distance|
+--------------+-------+------------------+
|greater_than_0| 424773| 3.470743355446981|
|  zero_or_null|2539795|3.6824345463125323|
+--------------+-------+------------------+

== Physical Plan ==
AdaptiveSparkPlan (16)
+- HashAggregate (15)
   +- Exchange (14)
      +- HashAggregate (13)
         +- Project (12)
            +- BroadcastHashJoin LeftOuter BuildRight (11)
               :- Scan In-memory table default.taxi_trips (1)
               :     +- InMemoryRelation (2)
               :           +- * ColumnarToRow (4)
               :              +- Scan parquet spark_catalog.default.taxi_trips (3)
               +- BroadcastExchange (10)
                  +- Filter (9)
                     +- Scan In-memory table default.weather (5)
                           +- InMemoryRelation (6)
                                 +- * ColumnarToRow (8)
                                    +- Scan parquet s

### Query 2.3

In [26]:
res = spark.sql(query_2_3())
res.show()
res.explain(mode="formatted")

+-----------+-------+
|measurement|  trips|
+-----------+-------+
|       NULL|2945398|
|        1.3|     20|
|        1.6|     13|
|        1.7|     31|
|        1.8|     41|
|        1.9|     19|
|        2.0|     24|
|        2.1|    135|
|        2.2|     39|
|        2.3|     36|
|        2.4|     51|
|        2.5|    137|
|        2.6|     93|
|        2.7|     84|
|        2.8|     27|
|        2.9|    141|
|        3.0|     62|
|        3.1|     89|
|        3.2|    115|
|        3.3|     71|
+-----------+-------+
only showing top 20 rows
== Physical Plan ==
AdaptiveSparkPlan (34)
+- Sort (33)
   +- Exchange (32)
      +- HashAggregate (31)
         +- Exchange (30)
            +- HashAggregate (29)
               +- Project (28)
                  +- BroadcastHashJoin LeftOuter BuildRight (27)
                     :- Project (12)
                     :  +- BroadcastHashJoin LeftOuter BuildRight (11)
                     :     :- Scan In-memory table default.taxi_trips (1)
     

### Query 2.5

In [27]:
res = spark.sql(query_2_5())
res.show()
res.explain(mode="formatted")

+---+----+-----+
|day|hour|trips|
+---+----+-----+
|Fri|  18|29050|
|Fri|  17|28034|
|Fri|  19|26656|
|Fri|  16|25645|
|Fri|  15|25578|
|Fri|  14|24746|
|Fri|  22|24015|
|Fri|  13|22190|
|Fri|  23|22100|
|Fri|  21|21976|
|Fri|  20|21168|
|Fri|  12|21024|
|Fri|  11|19852|
|Fri|  10|19559|
|Fri|   9|18192|
|Fri|   8|17323|
|Fri|   7|13043|
|Fri|   0| 8804|
|Fri|   6| 6283|
|Fri|   1| 4805|
+---+----+-----+
only showing top 20 rows
== Physical Plan ==
AdaptiveSparkPlan (11)
+- Sort (10)
   +- Exchange (9)
      +- HashAggregate (8)
         +- Exchange (7)
            +- HashAggregate (6)
               +- Project (5)
                  +- Scan In-memory table default.taxi_trips (1)
                        +- InMemoryRelation (2)
                              +- * ColumnarToRow (4)
                                 +- Scan parquet spark_catalog.default.taxi_trips (3)


(1) Scan In-memory table default.taxi_trips
Output [1]: [pu_datetime#13778]
Arguments: [pu_datetime#13778]

(2) InMemoryRel

### Query 2.6

In [28]:
res = spark.sql(query_2_6())
res.show()
res.explain(mode="formatted")

+-----+-------+
|month|  trips|
+-----+-------+
|  Dec|     12|
|  Feb|      3|
|  Jan|2964553|
+-----+-------+

== Physical Plan ==
AdaptiveSparkPlan (11)
+- Sort (10)
   +- Exchange (9)
      +- HashAggregate (8)
         +- Exchange (7)
            +- HashAggregate (6)
               +- Project (5)
                  +- Scan In-memory table default.taxi_trips (1)
                        +- InMemoryRelation (2)
                              +- * ColumnarToRow (4)
                                 +- Scan parquet spark_catalog.default.taxi_trips (3)


(1) Scan In-memory table default.taxi_trips
Output [1]: [pu_datetime#14044]
Arguments: [pu_datetime#14044]

(2) InMemoryRelation
Arguments: [pu_datetime#14044, do_datetime#14045, pu_location_id#14046, do_location_id#14047, fare_amount#14048, trip_distance#14049], StorageLevel(disk, memory, deserialized, 1 replicas)

(3) Scan parquet spark_catalog.default.taxi_trips
Output [6]: [pu_datetime#3844, do_datetime#3845, pu_location_id#3846, do_lo